# Chapter 16: Working with Categorical Data, String Operations, and Text Cleaning

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [12]:
import pandas as pd
import numpy as np

# Working with Categorical Data, String Operations, and Text Cleaning

## Introduction

Categorical data represents a common data type in real-world datasets—think product categories, survey responses, or geographic regions. Pandas provides powerful tools to work with categorical data efficiently, alongside comprehensive string manipulation capabilities. This chapter explores how to leverage these features for cleaner, more performant data analysis workflows.

By the end of this chapter, you will be able to:

- Create and manipulate categorical data in pandas
- Use the `.str` accessor for vectorized string operations
- Build text cleaning workflows for messy real-world data
- Understand when and why to use categorical types for performance
- Save and restore categorical data across different file formats

---

## Understanding Categorical Data

### What Are Categories?

Categorical data consists of a limited, fixed set of possible values. Unlike regular object dtypes that store each value independently, categorical data stores values more efficiently by maintaining a list of unique categories and referencing them by integer codes.

Consider a column storing the color of products. If you have one million rows but only three distinct colors, an object dtype stores the full string for every row. A categorical dtype stores the three strings once and uses small integers to point to them.

In [13]:
import pandas as pd
import numpy as np

# Create a Series with repeated string values
colors = pd.Series(['red', 'blue', 'red', 'green', 'blue', 'red', 'blue'])
print(f"Object dtype memory: {colors.memory_usage(deep=True)} bytes")

# Convert to categorical
colors_cat = colors.astype('category')
print(f"Category dtype memory: {colors_cat.memory_usage(deep=True)} bytes")

Object dtype memory: 553 bytes
Category dtype memory: 426 bytes


### When to Use Categories

Consider converting to categorical type when:

- A column contains many repeated string values
- You want to enforce specific allowed values
- You need ordered categories (for example, `"low" < "medium" < "high"`)
- You are working with large datasets where memory optimization matters

---

## Creating Categorical Data

There are several ways to create categorical data in pandas.

In [14]:
import pandas as pd
import numpy as np

# Method 1: Convert an existing Series
df = pd.DataFrame({
    'product': ['laptop', 'phone', 'laptop', 'tablet', 'phone'],
    'sales': [1200, 800, 1100, 450, 850]
})
df['product'] = df['product'].astype('category')

# Method 2: Specify categories explicitly during creation
ratings = pd.Categorical(
    ['good', 'excellent', 'fair', 'good', 'excellent'],
    categories=['poor', 'fair', 'good', 'excellent'],
    ordered=True
)
df_ratings = pd.DataFrame({'rating': ratings})

# Method 3: Use pd.cut() to bin continuous data into categories
ages = [5, 15, 25, 35, 45, 55, 65]
age_groups = pd.cut(
    ages,
    bins=[0, 18, 35, 50, 100],
    labels=['child', 'young adult', 'adult', 'senior']
)
print(age_groups)

['child', 'child', 'young adult', 'young adult', 'adult', 'senior', 'senior']
Categories (4, object): ['child' < 'young adult' < 'adult' < 'senior']


---

## Working with Categorical Properties

### Accessing Category Information

Once you have categorical data, you can inspect and manipulate its properties using the `.cat` accessor:

In [15]:
import pandas as pd

df = pd.DataFrame({
    'department': pd.Categorical(
        ['Sales', 'IT', 'HR', 'Sales', 'IT'],
        categories=['HR', 'IT', 'Sales', 'Marketing']
    )
})

# View the defined categories
print(df['department'].cat.categories)
# Index(['HR', 'IT', 'Sales', 'Marketing'], dtype='object')

# Check whether the categories are ordered
print(df['department'].cat.ordered)
# False

# Count the number of defined categories
print(len(df['department'].cat.categories))
# 4

Index(['HR', 'IT', 'Sales', 'Marketing'], dtype='object')
False
4


### Renaming Categories

You can rename categories without changing the underlying data values:

In [16]:
import pandas as pd

satisfaction = pd.Series(
    pd.Categorical([1, 2, 3, 2, 1], categories=[1, 2, 3])
)

satisfaction = satisfaction.cat.rename_categories(
    ['low', 'medium', 'high']
)

print(satisfaction)

0       low
1    medium
2      high
3    medium
4       low
dtype: category
Categories (3, object): ['low', 'medium', 'high']


You can also rename using a mapping with `rename_categories`:

In [17]:
satisfaction = pd.Series(
    pd.Categorical(['a', 'b', 'b', 'a', 'c']),
    index=['customer_1', 'customer_2', 'customer_3', 'customer_4', 'customer_5']
)
satisfaction = satisfaction.cat.rename_categories({'a': 'poor', 'b': 'average', 'c': 'excellent'})
print(satisfaction)

customer_1         poor
customer_2      average
customer_3      average
customer_4         poor
customer_5    excellent
dtype: category
Categories (3, object): ['poor', 'average', 'excellent']


### Reordering Categories

For ordered categorical data, reordering affects comparisons and sorting:

In [18]:
performance = pd.Series(
    pd.Categorical(
        ['B', 'A', 'C', 'B', 'A'],
        categories=['C', 'B', 'A'],
        ordered=True
    )
)

# Reorder so that A is highest
performance = performance.cat.set_categories(['A', 'B', 'C'], ordered=True)
print(performance.sort_values())

1    A
4    A
0    B
3    B
2    C
dtype: category
Categories (3, object): ['A' < 'B' < 'C']


### Adding and Removing Categories

In [19]:
status = pd.Series(
    pd.Categorical(['active', 'inactive', 'active'],
                   categories=['active', 'inactive'])
)

# Add new categories (no data changes yet)
status = status.cat.add_categories(['pending', 'archived'])
print(status.cat.categories)
# Index(['active', 'inactive', 'pending', 'archived'], dtype='object')

# Remove categories that have no data
status = status.cat.remove_unused_categories()
print(status.cat.categories)
# Index(['active', 'inactive'], dtype='object')

Index(['active', 'inactive', 'pending', 'archived'], dtype='object')
Index(['active', 'inactive'], dtype='object')


---

## Setting Values in Categorical Columns

When working with categorical data, you can only assign values that exist in the defined categories. Attempting to assign an unknown value raises a `ValueError`:

In [ ]:
df = pd.DataFrame({
    'status': pd.Categorical(
        ['approved', 'pending', 'approved'],
        categories=['approved', 'pending', 'rejected']
    )
})

# This works — 'rejected' is already a defined category
df.loc[1, 'status'] = 'rejected'
print(df)

# This fails — 'cancelled' is not in the categories
try:
    df.loc[0, 'status'] = 'cancelled'
except (ValueError, TypeError) as e:
    print(f"Error: {e}")

To assign a value outside the existing categories, expand the category list first:

In [24]:
df['status'] = df['status'].cat.add_categories(['cancelled'])
df.loc[0, 'status'] = 'cancelled'
print(df)

      status
0  cancelled
1   rejected
2   approved


This constraint is actually a feature: it prevents typos and invalid values from silently entering your data.

---

## String Operations with the `.str` Accessor

The `.str` accessor provides vectorized string operations on a Series. It works on both object dtype and categorical dtype columns.

### Basic String Methods

In [25]:
emails = pd.Series([
    'john.doe@company.com',
    'jane.smith@company.com',
    'bob.wilson@company.com'
])

# Extract the domain portion
domains = emails.str.split('@').str[1]
print(domains)

# Convert to uppercase
print(emails.str.upper())

# Check whether a substring is present
# Note: contains() treats '.' as a regex metacharacter by default
has_dot = emails.str.contains(r'\.', regex=True)
print(has_dot)

# Get string length
lengths = emails.str.len()
print(lengths)

0    company.com
1    company.com
2    company.com
dtype: object
0      JOHN.DOE@COMPANY.COM
1    JANE.SMITH@COMPANY.COM
2    BOB.WILSON@COMPANY.COM
dtype: object
0    True
1    True
2    True
dtype: bool
0    20
1    22
2    22
dtype: int64


### Common String Methods

In [26]:
text_data = pd.Series(['Hello World', 'PANDAS', 'data science', '  spaces  '])

# Case conversion
print(text_data.str.lower())
print(text_data.str.upper())
print(text_data.str.title())

# Trimming whitespace and padding
print(text_data.str.strip())
print(text_data.str.pad(width=20, side='left', fillchar='-'))

# Finding and replacing
print(text_data.str.find('a'))
print(text_data.str.replace('World', 'Universe', regex=False))

# String length
print(text_data.str.len())

0     hello world
1          pandas
2    data science
3        spaces  
dtype: object
0     HELLO WORLD
1          PANDAS
2    DATA SCIENCE
3        SPACES  
dtype: object
0     Hello World
1          Pandas
2    Data Science
3        Spaces  
dtype: object
0     Hello World
1          PANDAS
2    data science
3          spaces
dtype: object
0    ---------Hello World
1    --------------PANDAS
2    --------data science
3    ----------  spaces  
dtype: object
0   -1
1   -1
2    1
3    4
dtype: int64
0    Hello Universe
1            PANDAS
2      data science
3          spaces  
dtype: object
0    11
1     6
2    12
3    10
dtype: int64


### Pattern Matching and Extraction

In [27]:
product_codes = pd.Series([
    'PROD-2023-001',
    'PROD-2023-002',
    'PROD-2024-001',
    'SERV-2023-001'
])

# Extract the year using a regex capture group
years = product_codes.str.extract(r'(\d{4})', expand=False)
print(years)

# Extract all digit sequences (returns a MultiIndex DataFrame)
codes = product_codes.str.extractall(r'(\d+)')
print(codes)

# Check whether the string starts with a pattern
is_product = product_codes.str.match(r'PROD')
print(is_product)

0    2023
1    2023
2    2024
3    2023
dtype: object
            0
  match      
0 0      2023
  1       001
1 0      2023
  1       002
2 0      2024
  1       001
3 0      2023
  1       001
0     True
1     True
2     True
3    False
dtype: bool


### String Replacement and Cleaning

In [28]:
messy_data = pd.Series([
    '  John Doe  ',
    'jane_smith',
    'bob-wilson',
    'alice.johnson'
])

# Remove leading and trailing whitespace
cleaned = messy_data.str.strip()

# Replace underscores and hyphens with spaces
cleaned = cleaned.str.replace(r'[_\-.]', ' ', regex=True)

# Apply title case
cleaned = cleaned.str.title()
print(cleaned)

0         John Doe
1       Jane Smith
2       Bob Wilson
3    Alice Johnson
dtype: object


---

## Text Cleaning Workflows

### Handling Missing and Inconsistent Data

A common challenge is the same value appearing in multiple forms due to inconsistent data entry:

In [30]:
raw_data = pd.Series([
    'New York',
    'new york',
    'NEW YORK',
    'New york',
    None,
    'ny'
])

# Standardize to lowercase and remove surrounding whitespace
standardized = raw_data.str.lower().str.strip()

# Replace known abbreviations
replacements = {'ny': 'new york', 'ca': 'california'}
standardized = standardized.replace(replacements)

print(standardized)

0    new york
1    new york
2    new york
3    new york
4        None
5    new york
dtype: object


### Removing Special Characters

In [31]:
text_data = pd.Series([
    'Price: $99.99!',
    'Discount: 20% off!',
    'Contact: info@site.com'
])

# Keep only alphanumeric characters and spaces
cleaned = text_data.str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
print(cleaned)

0             Price 9999
1        Discount 20 off
2    Contact infositecom
dtype: object


### Cleaning Phone Numbers

Phone numbers are a classic example of data that arrives in many formats:

In [32]:
phones = pd.Series([
    '(555) 123-4567',
    '555.123.4567',
    '555-123-4567',
    '5551234567'
])

# Strip all non-digit characters
cleaned_phones = phones.str.replace(r'\D', '', regex=True)
print(cleaned_phones)

# Reformat consistently as (NXX) NXX-XXXX
formatted = cleaned_phones.str.replace(
    r'(\d{3})(\d{3})(\d{4})', r'(\1) \2-\3', regex=True
)
print(formatted)

0    5551234567
1    5551234567
2    5551234567
3    5551234567
dtype: object
0    (555) 123-4567
1    (555) 123-4567
2    (555) 123-4567
3    (555) 123-4567
dtype: object


### Splitting and Expanding

In [33]:
full_names = pd.Series([
    'John Michael Doe',
    'Jane Smith',
    'Robert Lee Wilson'
])

# Split on whitespace, limiting to one split to get first and remainder
name_parts = full_names.str.split(' ', n=1, expand=True)
name_parts.columns = ['first', 'last']
print(name_parts)

    first         last
0    John  Michael Doe
1    Jane        Smith
2  Robert   Lee Wilson


---

## Performance Considerations

When working with large Series containing many repeated string values, converting to categorical first can significantly improve both memory usage and operation speed.

In [34]:
import time

# Create a large Series with only three distinct values
large_series = pd.Series(
    np.random.choice(['category_a', 'category_b', 'category_c'], size=1_000_000)
)

# Approach 1: String operations directly on object dtype
start = time.time()
result1 = large_series.str.upper()
time1 = time.time() - start

# Approach 2: Convert to categorical first, then operate
start = time.time()
cat_series = large_series.astype('category')
result2 = cat_series.str.upper()
time2 = time.time() - start

print(f"Direct operation:    {time1:.4f}s")
print(f"Categorical first:   {time2:.4f}s")

Direct operation:    0.0553s
Categorical first:   0.0208s


Key guidelines:

- **Memory savings**: Categorical dtype uses significantly less memory than object dtype for columns with many repeated values.
- **Operation speed**: String operations on categorical data are faster because pandas applies the operation only to the unique categories, not to every row.
- **Trade-off**: There is slight overhead in maintaining the category index. The benefit grows with data size and the ratio of repeated values to unique values.
- **Rule of thumb**: Convert to categorical when the number of unique values is much smaller than the total number of rows.

---

## Saving and Loading Categorical Data

### CSV Considerations

CSV is a plain-text format and cannot store dtype metadata. When you save a categorical column to CSV and read it back, it becomes an object dtype:

In [35]:
df = pd.DataFrame({
    'quality': pd.Categorical(
        ['high', 'low', 'medium', 'high'],
        categories=['low', 'medium', 'high'],
        ordered=True
    ),
    'value': [100, 50, 75, 120]
})

# Save to CSV
df.to_csv('data.csv', index=False)

# Read back — quality is now object dtype
df_loaded = pd.read_csv('data.csv')
print(df_loaded.dtypes)

# Restore the categorical type and ordering
df_loaded['quality'] = pd.Categorical(
    df_loaded['quality'],
    categories=['low', 'medium', 'high'],
    ordered=True
)
print(df_loaded.dtypes)

quality    object
value       int64
dtype: object
quality    category
value         int64
dtype: object


### Parquet and HDF5

For workflows where preserving categorical information matters, use Parquet or HDF5:

In [ ]:
# Parquet preserves categorical dtype (requires pyarrow or fastparquet)
df.to_parquet('data.parquet')
df_parquet = pd.read_parquet('data.parquet')
print(df_parquet.dtypes)  # quality is still Categorical

# HDF5 also preserves categorical dtype (requires tables package)
df.to_hdf('data.h5', key='df', mode='w', format='table')
df_hdf = pd.read_hdf('data.h5', 'df')
print(df_hdf.dtypes)  # quality is still Categorical

Parquet is generally preferred for production pipelines because it is columnar, compressed, and widely supported across data tools.

---

## Practical Example: Customer Survey Data Cleaning

The following example brings together everything covered in this chapter. Starting from raw, inconsistent survey data, we clean text, standardize categories, and produce a tidy DataFrame ready for analysis.

In [37]:
import pandas as pd
import numpy as np

# Raw survey data with inconsistent formatting
survey_data = pd.DataFrame({
    'customer_id': range(1, 6),
    'name': ['  John Doe  ', 'jane_smith', 'BOB WILSON', 'alice-johnson', 'Carol White'],
    'satisfaction': ['  very good  ', 'GOOD', 'bad', '  very good  ', 'good'],
    'product': ['laptop', 'phone', 'tablet', 'laptop', 'PHONE'],
    'region': ['North', 'South', 'North', 'East', 'South']
})

df = survey_data.copy()

# --- Clean the name column ---
df['name'] = (
    df['name']
    .str.strip()
    .str.replace(r'[_\-]', ' ', regex=True)
    .str.title()
)

# --- Standardize and categorize satisfaction ---
df['satisfaction'] = (
    df['satisfaction']
    .str.strip()
    .str.lower()
)
df['satisfaction'] = pd.Categorical(
    df['satisfaction'],
    categories=['bad', 'good', 'very good'],
    ordered=True
)

# --- Standardize and categorize product ---
df['product'] = df['product'].str.lower().astype('category')

# --- Categorize region ---
df['region'] = df['region'].astype('category')

print(df)
print()
print(df.dtypes)
print()
print(df.memory_usage(deep=True))

   customer_id           name satisfaction product region
0            1       John Doe    very good  laptop  North
1            2     Jane Smith         good   phone  South
2            3     Bob Wilson          bad  tablet  North
3            4  Alice Johnson    very good  laptop   East
4            5    Carol White         good   phone  South

customer_id        int64
name              object
satisfaction    category
product         category
region          category
dtype: object

Index           128
customer_id      40
name            337
satisfaction    300
product         301
region          298
dtype: int64


Notice how the pipeline follows a consistent pattern: clean the raw text first (strip, lowercase, replace), then convert to categorical. Applying `.astype('category')` before cleaning would raise errors because the messy values would become separate categories.

---

## Summary

Working with categorical data and text in pandas enables:

- **Memory efficiency**: Categorical dtype uses less memory than object dtype for columns with repeated values.
- **Performance**: String operations on categorical data are faster for large datasets because pandas operates on unique categories rather than every row.
- **Data integrity**: Defined categories enforce valid values and prevent silent data entry errors.
- **Flexibility**: The `.str` accessor provides powerful, vectorized text manipulation without writing loops.
- **Consistency**: Standardized text cleaning pipelines ensure data quality before analysis.

The key workflow to remember is: **clean text first, then convert to categorical**. This order ensures that inconsistent raw values are normalized before the category list is fixed.

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Creating and Exploring Categorical Data

Create a DataFrame representing a small product inventory with a 'category' column as a pandas Categorical type. Explore its properties including categories, codes, and whether it is ordered.

In [38]:
import pandas as pd

# Sample product data
data = {
    'product': ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y', 'Doohickey Z'],
    'category': ['Electronics', 'Electronics', 'Toys', 'Toys', 'Electronics'],
    'price': [29.99, 49.99, 9.99, 14.99, 99.99]
}

df = pd.DataFrame(data)

# TODO: Convert the 'category' column to a pandas Categorical dtype
df['category'] = ...

# TODO: Print the dtype of the 'category' column
print("dtype:", ...)

# TODO: Print the unique categories using the .cat accessor
print("Categories:", ...)

# TODO: Print the category codes (integer codes backing the categorical)
print("Codes:", ...)

# TODO: Print whether the categorical is ordered
print("Is ordered:", ...)

dtype: Ellipsis
Categories: Ellipsis
Codes: Ellipsis
Is ordered: Ellipsis


### Exercise 2: Ordered Categories and Comparisons

Create an ordered categorical column representing customer satisfaction levels. Use the ordering to filter rows where satisfaction is at or above a given level, and sort the DataFrame by satisfaction.

In [39]:
import pandas as pd

# Sample survey data
data = {
    'customer_id': [101, 102, 103, 104, 105, 106],
    'satisfaction': ['Good', 'Poor', 'Excellent', 'Fair', 'Good', 'Excellent']
}

df = pd.DataFrame(data)

# Define the ordered satisfaction levels from lowest to highest
levels = ['Poor', 'Fair', 'Good', 'Excellent']

# TODO: Convert 'satisfaction' to an ordered Categorical using the levels above
df['satisfaction'] = ...

# TODO: Sort the DataFrame by 'satisfaction' (ascending) and print it
df_sorted = ...
print("Sorted DataFrame:")
print(df_sorted)

# TODO: Filter rows where satisfaction is greater than or equal to 'Good' and print
high_satisfaction = ...
print("\nHigh satisfaction customers:")
print(high_satisfaction)

Sorted DataFrame:
Ellipsis

High satisfaction customers:
Ellipsis


### Exercise 3: String Operations with the .str Accessor

Given a DataFrame of raw customer names and email addresses, use the `.str` accessor to clean and extract information. Tasks include stripping whitespace, converting case, checking for a domain, and extracting usernames from emails.

In [40]:
import pandas as pd

# Raw customer data with messy strings
data = {
    'name': ['  alice johnson ', 'BOB SMITH', ' Carol White  ', 'dave BROWN', '  Eve Davis'],
    'email': ['alice@example.com', 'bob@gmail.com', 'carol@example.com', 'dave@yahoo.com', 'eve@example.com']
}

df = pd.DataFrame(data)

# TODO: Strip leading/trailing whitespace from 'name' and store back in 'name'
df['name'] = ...

# TODO: Convert 'name' to title case (e.g., 'Alice Johnson') and store back in 'name'
df['name'] = ...

# TODO: Create a new boolean column 'is_example_domain' that is True
#       when the email ends with '@example.com'
df['is_example_domain'] = ...

# TODO: Extract the username part of the email (everything before the '@')
#       and store it in a new column 'username'
df['username'] = ...

print(df)

       name              email is_example_domain  username
0  Ellipsis  alice@example.com          Ellipsis  Ellipsis
1  Ellipsis      bob@gmail.com          Ellipsis  Ellipsis
2  Ellipsis  carol@example.com          Ellipsis  Ellipsis
3  Ellipsis     dave@yahoo.com          Ellipsis  Ellipsis
4  Ellipsis    eve@example.com          Ellipsis  Ellipsis


### Exercise 4: Full Text Cleaning Workflow

Apply a complete text cleaning pipeline to a messy product reviews DataFrame. Clean product codes using regex, standardize a categorical sentiment column, and compare memory usage before and after converting the sentiment column to Categorical dtype.

In [41]:
import pandas as pd

# Messy product review data
data = {
    'review_id': [1, 2, 3, 4, 5, 6],
    'product_code': ['ABC-001 ', ' xyz-002', 'ABC-003', ' XYZ-004 ', 'abc-001', 'XYZ-002 '],
    'sentiment': ['Positive', 'negative', ' Neutral ', 'POSITIVE', 'Negative', ' neutral']
}

df = pd.DataFrame(data)

# TODO: Clean 'product_code' by stripping whitespace and converting to uppercase
df['product_code'] = ...

# TODO: Clean 'sentiment' by stripping whitespace and converting to title case
#       so values are consistently 'Positive', 'Negative', or 'Neutral'
df['sentiment'] = ...

# TODO: Print the value counts of the cleaned 'sentiment' column
print("Sentiment value counts:")
print(...)

# TODO: Record memory usage of 'sentiment' BEFORE converting to Categorical
mem_before = df['sentiment'].memory_usage(deep=True)

# TODO: Convert 'sentiment' to an ordered Categorical with
#       categories = ['Negative', 'Neutral', 'Positive']
df['sentiment'] = ...

# TODO: Record memory usage of 'sentiment' AFTER converting to Categorical
mem_after = df['sentiment'].memory_usage(deep=True)

print("\nCleaned DataFrame:")
print(df)
print(f"\nMemory before: {mem_before} bytes")
print(f"Memory after:  {mem_after} bytes")

Sentiment value counts:
Ellipsis

Cleaned DataFrame:
   review_id product_code sentiment
0          1     Ellipsis  Ellipsis
1          2     Ellipsis  Ellipsis
2          3     Ellipsis  Ellipsis
3          4     Ellipsis  Ellipsis
4          5     Ellipsis  Ellipsis
5          6     Ellipsis  Ellipsis

Memory before: 272 bytes
Memory after:  272 bytes


---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Creating and Exploring Categorical Data

In [42]:
import pandas as pd
import numpy as np

# Sample product data
data = {
    'product': ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y', 'Doohickey Z'],
    'category': ['Electronics', 'Electronics', 'Toys', 'Toys', 'Electronics'],
    'price': [29.99, 49.99, 9.99, 14.99, 99.99]
}

df = pd.DataFrame(data)

# Convert the 'category' column to a pandas Categorical dtype
df['category'] = pd.Categorical(df['category'])

# Print the dtype of the 'category' column
print("dtype:", df['category'].dtype)

# Print the unique categories using the .cat accessor
print("Categories:", df['category'].cat.categories)

# Print the category codes (integer codes backing the categorical)
print("Codes:", df['category'].cat.codes)

# Print whether the categorical is ordered
print("Is ordered:", df['category'].cat.ordered)

dtype: category
Categories: Index(['Electronics', 'Toys'], dtype='object')
Codes: 0    0
1    0
2    1
3    1
4    0
dtype: int8
Is ordered: False


### Solution 2: Ordered Categories and Comparisons

In [43]:
import pandas as pd
import numpy as np

# Sample survey data
data = {
    'customer_id': [101, 102, 103, 104, 105, 106],
    'satisfaction': ['Good', 'Poor', 'Excellent', 'Fair', 'Good', 'Excellent']
}

df = pd.DataFrame(data)

# Define the ordered satisfaction levels from lowest to highest
levels = ['Poor', 'Fair', 'Good', 'Excellent']

# Convert 'satisfaction' to an ordered Categorical using the levels above
df['satisfaction'] = pd.Categorical(df['satisfaction'], categories=levels, ordered=True)

# Sort the DataFrame by 'satisfaction' (ascending) and print it
df_sorted = df.sort_values('satisfaction')
print("Sorted DataFrame:")
print(df_sorted)

# Filter rows where satisfaction is greater than or equal to 'Good' and print
high_satisfaction = df[df['satisfaction'] >= 'Good']
print("\nHigh satisfaction customers:")
print(high_satisfaction)

Sorted DataFrame:
   customer_id satisfaction
1          102         Poor
3          104         Fair
0          101         Good
4          105         Good
2          103    Excellent
5          106    Excellent

High satisfaction customers:
   customer_id satisfaction
0          101         Good
2          103    Excellent
4          105         Good
5          106    Excellent


### Solution 3: String Operations with the .str Accessor

In [44]:
import pandas as pd
import numpy as np

# Raw customer data with messy strings
data = {
    'name': ['  alice johnson ', 'BOB SMITH', ' Carol White  ', 'dave BROWN', '  Eve Davis'],
    'email': ['alice@example.com', 'bob@gmail.com', 'carol@example.com', 'dave@yahoo.com', 'eve@example.com']
}

df = pd.DataFrame(data)

# Strip leading/trailing whitespace from 'name' and store back in 'name'
df['name'] = df['name'].str.strip()

# Convert 'name' to title case (e.g., 'Alice Johnson') and store back in 'name'
df['name'] = df['name'].str.title()

# Create a new boolean column 'is_example_domain' that is True
# when the email ends with '@example.com'
df['is_example_domain'] = df['email'].str.endswith('@example.com')

# Extract the username part of the email (everything before the '@')
# and store it in a new column 'username'
df['username'] = df['email'].str.split('@').str[0]

print(df)

            name              email  is_example_domain username
0  Alice Johnson  alice@example.com               True    alice
1      Bob Smith      bob@gmail.com              False      bob
2    Carol White  carol@example.com               True    carol
3     Dave Brown     dave@yahoo.com              False     dave
4      Eve Davis    eve@example.com               True      eve


### Solution 4: Full Text Cleaning Workflow

In [45]:
import pandas as pd
import numpy as np

# Messy product review data
data = {
    'review_id': [1, 2, 3, 4, 5, 6],
    'product_code': ['ABC-001 ', ' xyz-002', 'ABC-003', ' XYZ-004 ', 'abc-001', 'XYZ-002 '],
    'sentiment': ['Positive', 'negative', ' Neutral ', 'POSITIVE', 'Negative', ' neutral']
}

df = pd.DataFrame(data)

# Clean 'product_code' by stripping whitespace and converting to uppercase
df['product_code'] = df['product_code'].str.strip().str.upper()

# Clean 'sentiment' by stripping whitespace and converting to title case
# so values are consistently 'Positive', 'Negative', or 'Neutral'
df['sentiment'] = df['sentiment'].str.strip().str.title()

# Print the value counts of the cleaned 'sentiment' column
print("Sentiment value counts:")
print(df['sentiment'].value_counts())

# Record memory usage of 'sentiment' BEFORE converting to Categorical
mem_before = df['sentiment'].memory_usage(deep=True)

# Convert 'sentiment' to an ordered Categorical with
# categories = ['Negative', 'Neutral', 'Positive']
df['sentiment'] = pd.Categorical(
    df['sentiment'],
    categories=['Negative', 'Neutral', 'Positive'],
    ordered=True
)

# Record memory usage of 'sentiment' AFTER converting to Categorical
mem_after = df['sentiment'].memory_usage(deep=True)

print("\nCleaned DataFrame:")
print(df)
print(f"\nMemory before: {mem_before} bytes")
print(f"Memory after:  {mem_after} bytes")

Sentiment value counts:
sentiment
Positive    2
Negative    2
Neutral     2
Name: count, dtype: int64

Cleaned DataFrame:
   review_id product_code sentiment
0          1      ABC-001  Positive
1          2      XYZ-002  Negative
2          3      ABC-003   Neutral
3          4      XYZ-004  Positive
4          5      ABC-001  Negative
5          6      XYZ-002   Neutral

Memory before: 516 bytes
Memory after:  436 bytes
